# Model to analyze replacement of Russian imports by LNG

- no network conversion used
- analysis of the impact of LNG import capacity increasement

### Import packages

In [1]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import networkx as nx
import os
from infrastructure_model_data_import_functions import *
from infrastructure_model_input_data_prep_functions import *
from infrastructure_model_output_data_prep_functions import *

In [319]:
#set name for in- and output
scenario_name = 'IAEE_2025_run_2024_SP'

In [320]:
#output specifications
output_file_path = os.path.join('..', '..','01_data', '02_output_data', '02_unidirectional_results','01_paper_IAEE', '01_raw_results')
outout_file_name = '\outputs_' + scenario_name + '.xlsx'
#create full ouput paths
output_file_path_excel  = output_file_path + outout_file_name
full_output_file_path = os.path.abspath(os.path.join(os.getcwd(), output_file_path_excel))

### Import Data

In [321]:
# Specify the path to your Excel file
input_file_path_data_prep = os.path.join('..', '..','00_code_base', '07_data_prep')
input_file_path = os.path.join('..', '..','01_data', '01_input_data', '02_processed', '01_paper_IAEE')
excel_file_name = 'inputs_' + scenario_name + '.xlsx'
#data_case_russia_BASE.xlsx
#data_case_russia_peak_hour.xlsx

In [322]:
# Call the function to load the data
df_nodes, df_commodities, df_edges, df_parameter, df_supply_values = load_input_data(input_file_path, excel_file_name)

In [323]:
(network_nodes, commodities, edges, initial_capacities, max_capacities, 
 edge_cost, pipe_new_cost, pipe_conv_cost, pipe_conv_factor, supply_values, 
 node_values) = extract_network_data(
     df_nodes, df_commodities, df_edges, df_parameter, df_supply_values)

### Create input data structure

In [324]:
'''
Create slack nodes for all supply nodes
link a node to supply in case of shortage in the system to all supply nodes
the capacity is infinite but at an infinite (super high) cost
'''
(shortage_list, shortage_edges_list, 
 shortage_capacity_dict, shortage_cost_dict) = create_slack_nodes_and_links(
     node_values, commodities)

In [325]:
'''
Create slack nodes for all supply nodes
enable excess nodes that oversupply is also not a problem
'''
(excess_list, excess_edges_list, 
excess_capacity_dict, excess_cost_dict) = create_excess_nodes_for_supply(
    node_values, commodities)

Check the graph

In [326]:
#check that all nodes are connected via edges
check_graph_connectivity(edges)

The graph is connected.


In [327]:
#check that all nodes are connected via edges and identify not connected components
check_graph_connectivity_and_components(edges)

The graph is connected.


In [328]:
# Check if all nodes are connected
check_if_all_nodes_connected(network_nodes, edges)

All nodes are connected.


## Model

### Create model

In [329]:
# Create a new model
model = gp.Model("Grid_Transformation")

### Define parameters

In [330]:
# Parameters
commodities = commodities  # Commodity types
#real network elements
network_nodes = network_nodes # Nodes of the system
network_edges = edges  # Edges
initial_capacities = initial_capacities # Initial capacities
max_capacities = max_capacities  # Maximum capacities
costs_edge = edge_cost  # Cost to transport from node to node
capacity_new_cost = pipe_new_cost  # Cost to increase capacity
capacity_change_cost = pipe_conv_cost  # Cost to increase capacity
node_value = node_values #contains supply and demand values

#implement factor to adjust capacity when conversion from methane to hydrogen
#TODO Implement it from the input file and use a correct factor
conversion_factor = pipe_conv_factor

#slack parameters shortage
shortage_nodes = shortage_list
shortage_edges = shortage_edges_list
shortage_capacities = shortage_capacity_dict
shortage_cost = shortage_cost_dict

#slack parameters excess
excess_nodes = excess_list
excess_edges = excess_edges_list
excess_capacities = excess_capacity_dict
excess_cost = excess_cost_dict

#complete network of the model
all_edges = network_edges + excess_edges_list + shortage_edges_list

### Define decision variables

In [331]:
# Decision variables
x_flow = {} #flow of commodity on an edge
x_flow_shortage = {}
x_flow_excess = {}
y_new_cap = {} #new build capacity for a commodity on an edge between two edges
z_conv_cap = {} #capacity of a commodity converted on an edge between two nodes
Change = {} # Binary variable for switching

for commodity in commodities:
    x_flow[commodity] = {}
    x_flow_shortage[commodity] = {}
    x_flow_excess[commodity] = {}
    y_new_cap[commodity] = {}
    z_conv_cap[commodity] = {}
    Change[commodity] = {}
    for edge in all_edges:
        x_flow[commodity][edge] = model.addVar(vtype=GRB.CONTINUOUS, name=f"x_flow_{commodity}_{edge[0]}_{edge[1]}")
        y_new_cap[commodity][edge] = model.addVar(vtype=GRB.CONTINUOUS, name=f"y_new_cap{commodity}_{edge[0]}_{edge[1]}")
        z_conv_cap[commodity][edge] = model.addVar(vtype=GRB.CONTINUOUS, name=f"z_conv_cap{commodity}_{edge[0]}_{edge[1]}")
        Change[commodity][edge] = model.addVar(vtype=GRB.BINARY, name=f"change_{commodity}_{edge[0]}_{edge[1]}")
    for edge in shortage_edges:
        x_flow_shortage[commodity][edge] = model.addVar(vtype=GRB.CONTINUOUS, name=f"x_flow_shortage_{commodity}_{edge[0]}_{edge[1]}")
    for edge in excess_edges:
        x_flow_excess[commodity][edge] = model.addVar(vtype=GRB.CONTINUOUS, name=f"x_flow_excess_{commodity}_{edge[0]}_{edge[1]}")
model.update()

In [332]:
network_edges_inc_excess = network_edges + excess_edges_list
network_edges_inc_shortage = network_edges + shortage_edges_list

### Define objective and constraints

In [333]:
# Objective function (minimize total transportation cost + cost to increase and convert capacity)
model.setObjective(
    gp.quicksum(x_flow[commodity][edge] * costs_edge[commodity][f"{edge[0]}{edge[1]}"] 
                for commodity in commodities for edge in network_edges) +
    gp.quicksum(y_new_cap[commodity][edge] * capacity_new_cost[commodity][f"{edge[0]}{edge[1]}"] for commodity in commodities for edge in network_edges) +
    #gp.quicksum(z_conv_cap[commodity][edge] * capacity_change_cost[commodity][f"{edge[0]}{edge[1]}"] for commodity in commodities for edge in network_edges),
    gp.quicksum(x_flow_shortage[commodity][edge] * shortage_cost[commodity][f"{edge[0]}{edge[1]}"] 
                for commodity in commodities for edge in shortage_edges)+
    gp.quicksum(x_flow_excess[commodity][edge] * excess_cost[commodity][f"{edge[0]}{edge[1]}"] 
                for commodity in commodities for edge in excess_edges),
    GRB.MINIMIZE
)

# Constraints

#inflow of a node must equal the outflow of a node
for node in network_nodes:  
    for commodity in commodities:  
        # Flow conservation constraint for the current node and commodity
        model.addConstr(gp.quicksum(x_flow[commodity][edge] for edge in all_edges if edge[1] == node) 
                                    - gp.quicksum(x_flow[commodity][edge] for edge in network_edges_inc_excess if edge[0] == node)
                                    #substraction of network_edges_inc_shortage not necessary as network_edges_inc_excess
                                    #covers all outgoing flows
                                    + node_value[commodity][node] 
                                    == 0, f"flow_constraint_{commodity}_{node}")
#implement excess nodes
for node in excess_nodes:  
    for commodity in commodities:  
        # Flow conservation constraint for the current node and commodity
        model.addConstr(gp.quicksum(x_flow[commodity][edge] for edge in all_edges if edge[1] == node) 
                                    #- gp.quicksum(x_flow[commodity][edge] for edge in all_edges if edge[1] == node) 
                                    >= 0, f"flow_constraint_{commodity}_{node}")

#excess node at the sources to avoid infeasible problems.        
for node in excess_nodes:
    for commodity in commodities: 
        model.addConstr(gp.quicksum(x_flow[commodity][edge] for edge in excess_edges if edge[1] == node) >= 0,
                        f"supply_excess_{commodity}_{node}")
        
# Add constraint for equality between x_flow and x_flow_excess for the same edge
for commodity in commodities:
    for edge in excess_edges:
        # Ensure equality for the corresponding edges
        model.addConstr(x_flow[commodity][edge] == x_flow_excess[commodity][edge], 
                        f"equality_flow_excess_constraint_{commodity}_{edge}")
        
#Capacity constraint for excess flow
for commodity in commodities:
    for edge in excess_edges:
        model.addConstr(x_flow_excess[commodity][edge] 
                        <= 1000000, 
                        f"capacity_limit_excess{commodity}_{edge[0]}_{edge[1]}")

#implement shortage nodes
for node in shortage_nodes:  
    for commodity in commodities:  
        # Flow conservation constraint for the current node and commodity
        model.addConstr(gp.quicksum(x_flow[commodity][edge] for edge in all_edges if edge[1] == node) 
                                    #- gp.quicksum(x_flow[commodity][edge] for edge in all_edges if edge[1] == node) 
                                    >= 0, f"flow_constraint_shortage_{commodity}_{node}")

#shortage node at the sources to avoid infeasible problems.        
for node in shortage_nodes:
    for commodity in commodities: 
        model.addConstr(gp.quicksum(x_flow[commodity][edge] for edge in shortage_edges if edge[1] == node) >= 0,
                        f"supply_shortage_{commodity}_{node}")

# Add constraint for equality between x_flow and x_flow_shortage for the same edge
for commodity in commodities:
    for edge in shortage_edges:
        # Ensure equality for the corresponding edges
        model.addConstr(x_flow[commodity][edge] == x_flow_shortage[commodity][edge], 
                        f"equality_flow_shortage_constraint_{commodity}_{edge}")

#Capacity constraint for shortage flow
for commodity in commodities:
    for edge in shortage_edges:
        model.addConstr(x_flow_shortage[commodity][edge] 
                        <= 1000000, 
                        f"capacity_limit_shortage{commodity}_{edge[0]}_{edge[1]}")

#Capacity constraint for flow
for commodity in commodities:
    for edge in network_edges:
        model.addConstr(x_flow[commodity][edge] 
                        <= y_new_cap[commodity][edge] + z_conv_cap[commodity][edge] #+ initial_capacities[commodity][f"{edge[0]}{edge[1]}"]
                        , f"used_capacity_{commodity}_{edge[0]}_{edge[1]}")


# Capacity constraint for maximal capacity
for commodity in commodities:
    for edge in network_edges:
        model.addConstr(y_new_cap[commodity][edge] + z_conv_cap[commodity][edge] 
                        <= max_capacities[commodity][f"{edge[0]}{edge[1]}"], f"capacity_{commodity}_{edge[0]}_{edge[1]}")

#Constraints for edge conversion
for edge in network_edges:
    model.addConstr(Change[commodities[0]][edge] + Change[commodities[1]][edge] == 1, f"switching_constraint_{edge[0]}_{edge[1]}")

for commodity in commodities:
    for edge in network_edges:
        model.addConstr(initial_capacities[commodities[0]][f"{edge[0]}{edge[1]}"] * Change[commodity][edge] #* conversion_factor[commodity][node]
                        == z_conv_cap[commodity][edge], f"changed_capacity_{commodity}_{edge[0]}_{edge[1]}")

#Constraing no negative flow
for commodity in commodities:
    for edge in all_edges:
        model.addConstr(x_flow[commodity][edge] >= 0, f"non_negativity_x_{commodity}_{edge[0]}_{edge[1]}")

### Optimize the model

In [334]:
# Optimize the model
model.optimize()

Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (win64 - Windows 11+.0 (26100.2))

CPU model: Intel(R) Core(TM) i5-8265U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 5130 rows, 4414 columns and 10558 nonzeros
Model fingerprint: 0x9d4a0149
Variable types: 3360 continuous, 1054 integer (1054 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+07]
  Objective range  [7e+01, 1e+08]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+07]
Presolve removed 5034 rows and 3963 columns
Presolve time: 0.00s
Presolved: 96 rows, 451 columns, 806 nonzeros
Variable types: 451 continuous, 0 integer (0 binary)

Root relaxation: objective 1.781876e+12, 170 iterations, 0.02 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

*    0     0               0    1.

### Results processing

In [335]:
#check model status
check_optimization_status(model)

Optimal solution found!


In [336]:
#call the function to store the results into DataFrames
results_df, excess_df, shortage_df = store_optimization_results(
    model,
    commodities,
    network_edges,
    excess_edges,
    shortage_edges,
    x_flow,
    y_new_cap,
    Change,
    z_conv_cap,
    x_flow_excess,
    x_flow_shortage
)

Get results

In [337]:
# Call the function to filter the DataFrames for each commodity
commodity_dfs = filter_commodities_to_dataframe(results_df, commodities)
methane_rows_df = commodity_dfs.get('Methane')

In [338]:
#create a duplicate for further operations
methane_results_df = methane_rows_df

In [339]:
#get initial capacities but in another format than above
initial_capacities_separator_data = create_initial_capacities_separator_dict(df_parameter)
initial_capacities_data = initial_capacities_separator_data

In [340]:
#get the share of use for each infrastructure element
methane_results_df = add_share_column(methane_results_df, initial_capacities_data)
methane_results_df

,Commodity,Edge,Flow,Share
0,Methane,"(DE, LU)",0.000000e+00,0.000000
1,Methane,"(LT, PL)",0.000000e+00,0.000000
2,Methane,"(RU, UA)",0.000000e+00,0.000000
3,Methane,"(BE, NL)",0.000000e+00,0.000000
4,Methane,"(HR, SI)",0.000000e+00,0.000000
...,...,...,...,...
423,Methane,"(IN_Prod, IN)",3.085902e+05,0.030859
424,Methane,"(AS_Prod, AS)",1.259517e+06,0.125952
425,Methane,"(CR_Prod, CR)",1.828659e+06,0.182866
426,Methane,"(EG_Prod, EG)",5.578710e+05,0.055787


In [341]:
#aggregated share of capacity use
# Exclude flows and capacities from the following countries
excluded_countries = ['RU']
result_capacity_wo_ru_df = calculate_aggregated_capacity(methane_results_df, initial_capacities_data, excluded_countries)
result_capacity_wo_ru_df

,Commodity,Type,Total Flow,Total Capacity,Share
0,Methane,LNG,1.298409e+07,2.213233e+09,0.005867
1,Methane,Prod,3.679814e+07,5.399999e+08,0.068145
2,Methane,St,0.000000e+00,0.000000e+00,0.000000


In [342]:
#aggregated share of supply use (e.g., from potential production, LNG or storage as a source)
# Exclude capacities and flows from Russia ('RU')
excluded_countries = ['RU']
result_supply_wo_ru_df = calculate_aggregated_share_supply(methane_results_df, node_values, excluded_countries)
result_supply_wo_ru_df

,Commodity,Type,Total Flow,Total Potential Supply,Share
0,Methane,LNG,1.298409e+07,0.000000e+00,0.000000
1,Methane,Prod,3.679814e+07,3.897186e+07,0.944223
2,Methane,St,0.000000e+00,0.000000e+00,0.000000


In [343]:
search_element = 'RU'  # Example search element to look for in the 'Edge' column
commodity = 'Methane'  # Example commodity to filter by (optional)

# Call the function to filter the rows
filtered_df = filter_rows_by_search_element_and_commodity(results_df, 'Edge', search_element, commodity)
filtered_df

,Commodity,Edge,Flow,New Capacity,Switched,Changed Capacity
2,Methane,"(RU, UA)",0.000000e+00,0.0,1.0,1.992721e+06
21,Methane,"(RU, LV)",0.000000e+00,0.0,1.0,0.000000e+00
45,Methane,"(RU, BY)",1.846097e+05,0.0,1.0,4.392775e+05
46,Methane,"(RU, FI)",0.000000e+00,0.0,1.0,0.000000e+00
57,Methane,"(LV, RU)",0.000000e+00,0.0,1.0,0.000000e+00
70,Methane,"(LT, RU)",0.000000e+00,0.0,1.0,0.000000e+00
83,Methane,"(RU, DE)",0.000000e+00,0.0,1.0,0.000000e+00
88,Methane,"(RU, TR)",0.000000e+00,0.0,1.0,3.499985e+05
146,Methane,"(RU, RU_LNG_exp)",1.610744e+05,0.0,1.0,9.999999e+06
148,Methane,"(RU, CR)",0.000000e+00,0.0,1.0,7.933240e+05


In [344]:
search_element = 'USA'  # Example search element to look for in the 'Edge' column
commodity = 'Methane'  # Example commodity to filter by (optional)

# Call the function to filter the rows
filtered_df = filter_rows_by_search_element(methane_results_df, 'Edge', search_element)
filtered_df

,Commodity,Edge,Flow,Share
140,Methane,"(USA, USA_LNG_exp)",1.292080e+06,0.129208
149,Methane,"(USA, CM)",1.090448e+05,0.022322
154,Methane,"(CM, USA)",0.000000e+00,0.000000
160,Methane,"(USA_LNG_exp, LV_LNG_imp)",8.940323e+03,0.000894
162,Methane,"(USA_LNG_exp, CN_LNG_imp)",0.000000e+00,0.000000
166,Methane,"(USA_LNG_exp, HR_LNG_imp)",2.540200e+04,0.002540
170,Methane,"(USA_LNG_exp, TR_LNG_imp)",1.548626e+05,0.015486
176,Methane,"(USA_LNG_exp, EL_LNG_imp)",1.008315e+05,0.010083
200,Methane,"(USA_LNG_exp, IN_LNG_imp)",0.000000e+00,0.000000
201,Methane,"(USA_LNG_exp, FI_LNG_imp)",9.391820e+03,0.000939


In [345]:
#get just excess for methane out of the excess_df
methane_excess_df = excess_df[excess_df['Commodity'].str.contains('Methane', case=False)]
methane_excess_df

,Commodity,Edge,Flow
0,Methane,"(USA_Prod, USA_Prod_excess)",985653.388948
1,Methane,"(AU_Prod, AU_Prod_excess)",1000000.000000
2,Methane,"(ME_Prod, ME_Prod_excess)",194977.912176


In [346]:
'''still not working'''
#check for an unused share of capacities
#aggregated share of supply use (e.g., from potential production, LNG or storage as a source)
# Exclude capacities and flows
excluded_countries = ['ES']
methane_excess_share_df = calculate_aggregated_share_supply(methane_excess_df, node_values, excluded_countries)
methane_excess_share_df

,Commodity,Type,Total Flow,Total Potential Supply,Share
0,Methane,LNG,0.000000e+00,0.000000e+00,0.000000
1,Methane,Prod,2.180631e+06,4.470061e+07,0.048783
2,Methane,St,0.000000e+00,0.000000e+00,0.000000


In [347]:
methane_shortage_df = shortage_df[shortage_df['Commodity'].str.contains('Methane', case=False)]
methane_shortage_df

,Commodity,Edge,Flow
0,Methane,"(shortage_BA_Prod, BA_Prod)",6903.836523


In [ ]:
# Export to Excel with multiple sheets
with pd.ExcelWriter(full_output_file_path, engine='openpyxl') as writer:
    methane_results_df.to_excel(writer, index=False, sheet_name='flows_methane_edges')